In [ ]:
# Cell 1: (Install deps) Uncomment to install packages if running in a fresh environment
# !pip install pandas matplotlib seaborn python-dateutil tqdm
import unsloth
# Cell 2: Imports
import glob
import json
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from dateutil import parser as dateparser

# Cell 3: Locate JSONL files
messages_dir = Path("messages")
jsonl_files = sorted(glob.glob(str(messages_dir / "*.jsonl")))
print(f"Found {len(jsonl_files)} jsonl files")
jsonl_files[:10]  # preview

# Cell 4: Read and parse JSONL into a list of rows (one row per message)
rows = []

# Cell 5: Parse files (loop separated so it can be executed alone)
for fpath in tqdm(jsonl_files, desc="Reading jsonl files"):
    channel_name = Path(fpath).stem
    with open(fpath, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            # normalize common fields if present
            msg_id = obj.get("id") or obj.get("message_id")
            author = obj.get("author") or obj.get("user") or {}
            author_id = author.get("id") if isinstance(author, dict) else None
            author_name = None
            if isinstance(author, dict):
                author_name = author.get("name") or author.get("username") or author.get("display_name")
            content = obj.get("content") or obj.get("clean_content") or obj.get("text") or ""
            created_at = obj.get("created_at") or obj.get("timestamp") or obj.get("time") or obj.get("created")
            # try parse created_at to datetime
            ts = None
            if created_at:
                try:
                    ts = dateparser.parse(created_at)
                except Exception:
                    ts = None
            rows.append({
                "channel": channel_name,
                "msg_id": msg_id,
                "author_id": author_id,
                "author_name": author_name,
                "content": content,
                "ts": ts,
                "raw": obj,
            })

# Cell 6: Build DataFrame from parsed rows
import pandas as pd

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df.head(3)

# Cell 7: Deduplicate by msg_id when available
if "msg_id" in df.columns and df["msg_id"].notna().any():
    before = len(df)
    df = df.drop_duplicates(subset=["msg_id"]).reset_index(drop=True)
    after = len(df)
    print(f"Dropped {before-after} duplicate rows by msg_id; {after} remain")
else:
    print("No msg_id column or all missing msg_id; skipping dedupe")

In [ ]:
# Cell 8: Derived columns and normalize timestamp
# (kept separate so you can inspect before heavy transforms)
df["content"] = df["content"].fillna("")
df["content_len"] = df["content"].str.len()
df["word_count"] = df["content"].str.split().map(len)

df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["date"] = df["ts"].dt.date

df[["channel","author_name","content_len","word_count","ts"]].head(5)

# Cell 9: Basic summaries (channels and authors)
print("Messages by channel (top 10):")
print(df["channel"].value_counts().head(10))

print("\nMessages by author (top 10):")
print(df["author_name"].value_counts().head(10))

# Cell 10: Plot messages per day
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")

msgs_per_day = df.groupby("date").size().rename("count").reset_index()
plt.figure(figsize=(10,4))
plt.plot(msgs_per_day["date"], msgs_per_day["count"], marker=".")
plt.title("Messages per day")
plt.xlabel("Date")
plt.ylabel("Messages")
plt.tight_layout()
plt.show()

# Cell 11: Message length distribution
plt.figure(figsize=(8,4))
sns.histplot(df["content_len"].clip(upper=2000), bins=50)
plt.title("Message length distribution (chars)")
plt.xlabel("Chars")
plt.tight_layout()
plt.show()

# Cell 12: Top tokens (simple whitespace split) — top 25
from collections import Counter
cnt = Counter()
for s in df["content"].dropna():
    cnt.update(s.split())

common = cnt.most_common(25)
common

# Cell 13: Save a small preview CSV for quick inspection
preview_path = Path("messages") / "messages_preview.csv"
df.to_csv(preview_path, index=False)
print(f"Wrote preview CSV to {preview_path}")

In [ ]:
# Cell X: Identify and print high-volume days
if 'msgs_per_day' not in globals():
    msgs_per_day = df.groupby('date').size().rename('count').reset_index()

mean = msgs_per_day['count'].mean()
std = msgs_per_day['count'].std()
threshold = mean + 3 * std

high_days = msgs_per_day[msgs_per_day['count'] > threshold].sort_values('count', ascending=False)
print(f"Mean messages/day = {mean:.2f}, std = {std:.2f}, threshold = {threshold:.2f}")
print("\nDays above threshold (mean + 3*std):")
print(high_days.to_string(index=False))

print("\nTop 10 days overall:")
print(msgs_per_day.sort_values('count', ascending=False).head(10).to_string(index=False))

In [ ]:
# Cell Y: Print sample messages for the busiest day
# Ensure msgs_per_day exists
if 'msgs_per_day' not in globals():
    msgs_per_day = df.groupby('date').size().rename('count').reset_index()

busiest = msgs_per_day.sort_values('count', ascending=False).iloc[0]
busiest_day = busiest['date']
print(f"Busiest day: {busiest_day} with {busiest['count']} messages")

# Select messages from that day, sort by timestamp
day_msgs = df[df['date'] == busiest_day].sort_values('ts')

# Show up to 50 messages with truncated content for overview
from textwrap import shorten

preview = day_msgs[['ts','channel','author_name','content']].head(50).copy()
preview['content_preview'] = preview['content'].apply(lambda s: shorten(s.replace('\n',' '), width=200, placeholder='...'))

# Print as table-like text
for idx, row in preview.iterrows():
    ts = row['ts']
    ch = row['channel']
    a = row['author_name'] or '<unknown>'
    c = row['content_preview']
    print(f"{ts} | #{ch} | {a}: {c}\n")

In [ ]:
# Cell: Load OPENAI_API_KEY from .env using python-dotenv
try:
    from dotenv import load_dotenv
except Exception:
    raise RuntimeError('Please install python-dotenv (pip install python-dotenv) to load .env files')

from pathlib import Path
import os

env_path = Path('.') / '.env'
if not env_path.exists():
    print(f"Warning: {env_path} not found. Make sure .env exists in the notebook working directory.")
else:
    load_dotenv(dotenv_path=env_path)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if OPENAI_API_KEY:
    masked = OPENAI_API_KEY[:4] + '...' + OPENAI_API_KEY[-4:]
    print(f"Loaded OPENAI_API_KEY from .env (masked): {masked}")
else:
    print('OPENAI_API_KEY not found in environment after loading .env')

In [ ]:
# Cell Z: Few-shot ChatGPT emulation using sampled messages (openai>=1.0.0) with retry/backoff
import os
import time
from random import random

try:
    from openai import OpenAI
except Exception as e:
    raise RuntimeError("Please install openai>=1.0.0 (pip install --upgrade openai) to use this cell")

# Ensure API key is available
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found in environment. Export it or load it from .env before running this cell.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Select author to emulate (default: the most common author in the dataset)
emulate_author = None
if 'df' in globals():
    if emulate_author is None:
        try:
            emulate_author = df['author_name'].dropna().mode().iloc[0]
        except Exception:
            emulate_author = None

if emulate_author is None:
    emulate_author = input('Enter author_name to emulate (or press Enter to abort): ').strip() or None
if not emulate_author:
    raise RuntimeError('No author selected for emulation')

print(f"Emulating author: {emulate_author}")

# Sample example utterances from that author
if 'df' not in globals():
    raise RuntimeError('Dataframe `df` not found. Run earlier cells to load and parse messages.')

author_msgs = df[df['author_name'] == emulate_author]['content'].dropna().astype(str)
if author_msgs.empty:
    raise RuntimeError(f'No messages found for author: {emulate_author}')

sample_n = min(8, len(author_msgs))
examples = author_msgs.sample(sample_n, random_state=1).tolist()

# Truncate long examples
from textwrap import shorten
examples = [shorten(x.replace('\n',' '), width=400, placeholder='...') for x in examples]

# Build system prompt using examples as style reference
system_prompt = (
    f"You are a language assistant. When asked to reply, imitate the writing style of the examples below, which are authentic utterances from {emulate_author}. "
    "Mimic tone, sentence length, punctuation, and common expressions, but do not claim to be the real person. Use the examples only as style guidance.\n\n"
    "Examples:\n" + "\n---\n".join(f"{i+1}. {ex}" for i,ex in enumerate(examples))
)

# Prepare user test prompt (change as needed)
user_test_prompt = (
    "You are asked: 'How would you describe your day and plans for the weekend?' "
    "Respond as if you were the person represented by the examples above, keeping replies short and in-character."
)

# Compose chat messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_test_prompt}
]

# Retry/backoff configuration
max_retries = 5
initial_backoff = 1.0

model = "gpt-3.5-turbo"
print(f"Calling model {model} with {len(messages)} message(s) and {sample_n} examples...")

attempt = 0
while True:
    attempt += 1
    try:
        resp = client.chat.completions.create(model=model, messages=messages, max_tokens=256, temperature=0.8)
        # Extract reply text
        try:
            reply = resp.choices[0].message.content
        except Exception:
            reply = resp['choices'][0]['message']['content']
        print('\n=== Emulated Reply ===\n')
        print(reply.strip())
        break
    except Exception as e:
        # Try to interpret OpenAIError-like payloads
        err_str = str(e)
        # Inspect common patterns to detect insufficient_quota vs transient rate limits
        if 'insufficient_quota' in err_str or 'insufficient quota' in err_str:
            print('\nERROR: Insufficient quota detected.\n')
            print('OpenAI responded with an insufficient_quota error. This is not retriable. Please check your OpenAI billing and quota settings: https://platform.openai.com/account/usage')
            break
        if attempt >= max_retries:
            print(f"\nERROR: Failed after {attempt} attempts. Last error:\n{err_str}")
            break
        # Exponential backoff with jitter for retriable errors (HTTP 429 etc.)
        backoff = initial_backoff * (2 ** (attempt - 1))
        backoff = backoff * (0.75 + 0.5 * random())  # jitter
        print(f"Transient error (attempt {attempt}/{max_retries}): {err_str}\nRetrying in {backoff:.1f}s...")
        time.sleep(backoff)

# Optional: show the examples used (shortened)
print('\n=== Examples used for style reference ===\n')
for i,ex in enumerate(examples,1):
    print(f"{i}. {ex}\n")

In [ ]:

model_name = "meta-llama/Meta-Llama-3.1-8B"
model, tokenizer = unsloth.FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 4096,
    dtype = None,        # Auto-detect optimal dtype
    load_in_4bit = True  # QLoRA ⚡ enables training on small VRAM
)
